# Lab - Evaluation: From Two Scores to a Release Decision

Three tasks that turn the lecture's test set into what an architect needs before a prompt change ships: cases that represent the traffic, results by category, and a decision record with its cost.

| Task | What you learn |
| --- | --- |
| 1 | A test set with a category per case, and both sides of the boundary |
| 2 | Score by category: see where the gap is, not only that it exists |
| 3 | Decide: no category worse, total better, known gaps listed, cost priced |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"

PROMPT_V1 = (
    "You are ShopAssist, an online store's support assistant. "
    "Returns are accepted within 30 days of delivery. "
    "Begin your answer with YES or NO, then one sentence."
)
PROMPT_V2 = PROMPT_V1 + " Gift cards and underwear cannot be returned."

# What a release costs: the traffic and the input price (dollars per million tokens).
REQUESTS_PER_MONTH = 20_000 * 30
PRICE_IN = 2.00


def answer(system, text):
    """One question, one system prompt - the whole response, so usage comes along."""
    return client.messages.create(
        model=MODEL,
        max_tokens=100,
        output_config={"effort": "low"},
        system=system,
        messages=[{"role": "user", "content": text}],
    )

---

## Task 1 - A test set that covers every kind of case

In [ ]:
# ============================================================
# TASK 1 - A test set that covers every kind of case
# ============================================================
#
# WHAT TO DO
#   The six cases from the lecture now carry a category. Add
#   two cases the six do not have: a delivery one day past the
#   window, and an item the policy excludes that no prompt
#   mentions yet (earrings). Then count, per category, which
#   verdicts the set expects.
#
# WHY IT MATTERS
#   A score is only as good as the cases behind it. An
#   architect asks what the set represents: every kind of
#   question customers send, and both sides of every boundary.
#   A set with no case past the window cannot catch a prompt
#   that says yes to day 31.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Then the test set: six cases, each a customer question and
#   the expected verdict".
#
# HOW TO DO IT
#   Three blanks:
#
#       "NO", "boundary"    day 31 is one day too late
#       "NO", "excluded"    earrings cannot be returned
#       category            group the verdicts by category
# ============================================================

CATEGORIES = ["in window", "late", "excluded", "boundary"]

try:
    CASES = [
        ("I bought a jacket, delivered 20 days ago. Can I return it?", "YES", "in window"),
        ("The shoes arrived yesterday. I want to send them back.", "YES", "in window"),
        ("My order was delivered 45 days ago. Can I still return it?", "NO", "late"),
        ("I got a gift card last week. Can I return it?", "NO", "excluded"),
        ("The underwear I ordered came 3 days ago. Return possible?", "NO", "excluded"),
        ("Delivered 29 days ago, is a return still possible?", "YES", "boundary"),
        # TODO: replace each ... below with the value named beside it
        ("Delivered 31 days ago. Can I still send it back?", ..., ...),   # "NO", "boundary"
        ("Can I return earrings I got 5 days ago?", ..., ...),   # "NO", "excluded"
    ]
    
    def coverage(cases):
        """category -> the verdicts its cases expect"""
        by_category = {}
        for question, expected, category in cases:
            by_category.setdefault(..., set()).add(expected)   # category
        return by_category

    covered = coverage(CASES)
    if ... in covered or any(... in case for case in CASES):   # provided: a blank, said plainly
        raise TypeError("ellipsis")
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for category in CATEGORIES:
    print(f"{category:10} expects {sorted(covered.get(category, []))}")

check("eval_coverage", cases=CASES, categories=CATEGORIES)

---

## Task 2 - Score by category, not only in total

In [ ]:
# ============================================================
# TASK 2 - Score by category, not only in total
# ============================================================
#
# WHAT TO DO
#   Finish evaluate(): one result per case - did the reply
#   start with the expected verdict, and how many input tokens
#   did the request take. Then add the results up per category
#   and run the first prompt.
#
# WHY IT MATTERS
#   "Six out of eight" says that something is wrong; the table
#   by category says where. Here the gap sits in one category,
#   items the store does not take back, and only there. That is
#   what you fix, and what you watch on the next change.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The prompt has a gap, and the test found it in seconds".
#
# HOW TO DO IT
#   Three blanks:
#
#       expected                    the verdict the reply must start with
#       response.usage.input_tokens what the request cost
#       r["category"]               the row this result counts in
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    def evaluate(system):
        results = []
        for question, expected, category in CASES:
            response = answer(system, question)
            reply = response.content[0].text
            results.append({
                "question": question, "category": category, "expected": expected,
                "ok": reply.strip().upper().startswith(...),   # expected
                "input_tokens": ...,   # response.usage.input_tokens
            })
        return results
    
    def by_category(results):
        table = {}
        for r in results:
            passed, total = table.get(r["category"], (0, 0))
            table[...] = (passed + r["ok"], total + 1)   # r["category"]
        return table

    v1 = evaluate(PROMPT_V1)
    v1_table = by_category(v1)
    if ... in v1_table or any(r["input_tokens"] is ... for r in v1):   # provided: said plainly
        raise TypeError("ellipsis")
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for category, (passed, total) in v1_table.items():
    print(f"{category:10} {passed}/{total}")
print("v1 total:", sum(r["ok"] for r in v1), "/", len(v1))

check("eval_by_category", results=v1, table=v1_table, cases=CASES)

---

## Task 3 - Ship it? A release decision with its price

In [ ]:
# ============================================================
# TASK 3 - Ship it? A release decision with its price
# ============================================================
#
# WHAT TO DO
#   Run the second prompt and compare the tables. Write the
#   decision as a record: ship only if no category got worse and
#   the total went up. List the cases that still fail, and price
#   the longer prompt at this traffic.
#
# WHY IT MATTERS
#   The lecture's comparison - same cases, two prompts, two
#   scores - is the evidence. The decision record is what you
#   hand the team: what improved, what got worse, what is still
#   open, and what it costs. Six cases catch the gap you thought
#   of; the known gaps are the next cases to add.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Over time the set grows, with every real customer question
#   that went wrong".
#
# HOW TO DO IT
#   Three blanks:
#
#       v2_table[c][0]   the second prompt's passes in category c
#       not regressed and score_v2 > score_v1
#                        nothing worse, and better overall
#       "ok"             the cases the new prompt still fails
# ============================================================

try:
    v2 = evaluate(PROMPT_V2)
    v2_table = by_category(v2)
    # TODO: replace each ... below with the value named beside it
    regressed = [c for c in v1_table if ... < v1_table[c][0]]   # v2_table[c][0]
    improved = [c for c in v1_table if v2_table[c][0] > v1_table[c][0]]
    score_v1 = sum(r["ok"] for r in v1)
    score_v2 = sum(r["ok"] for r in v2)
    ship = ...   # not regressed and score_v2 > score_v1
    known_gaps = [r["question"] for r in v2 if not r[...]]   # "ok"

    if ship is ...:   # provided: a blank left above, said plainly
        raise TypeError("ellipsis")
    extra_tokens = (sum(r["input_tokens"] for r in v2) - sum(r["input_tokens"] for r in v1)) / len(v1)
    decision = {   # provided: the record you hand the team
        "ship": ship,
        "score": f"{score_v1}/{len(v1)} -> {score_v2}/{len(v2)}",
        "improved": improved,
        "regressed": regressed,
        "known_gaps": known_gaps,
        "extra_tokens_per_request": round(extra_tokens, 1),
        "extra_cost_per_month": round(extra_tokens * REQUESTS_PER_MONTH * PRICE_IN / 1_000_000, 2),
    }
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for key, value in decision.items():
    print(f"{key:24} {value}")

check("eval_decision", decision=decision, v1=v1, v2=v2)

---

## Done

Keep a test set that represents the traffic, by category and on both sides of every boundary. Compare prompts by category, ship only when nothing got worse, and write down what is still failing and what the change costs.